In [ ]:
%pip install -U gradio langchain langchain-community langchain-core sentence-transformers faiss-cpu mlflow pypdf databricks-langchain
dbutils.library.restartPython()


In [0]:
import os, json, traceback, gradio as gr
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import FAISS
try:
    from databricks_langchain import ChatDatabricks
except Exception:
    from langchain_community.chat_models import ChatDatabricks


In [0]:
CONFIG_PATH="dbfs:/Volumes/main/default/dino_pdfs/project/config/project_config.json"
config=json.loads(dbutils.fs.head(CONFIG_PATH,100000))
INDEX_URI=config["index_uri"]
CHAT_ENDPOINT=config["chat_endpoint"]
EMBEDDING_MODEL_NAME=config.get("embedding_model_name","sentence-transformers/all-MiniLM-L6-v2")
DEFAULT_TOP_K=int(config.get("top_k",4))
print(json.dumps(config,indent=2))


{
  "project_name": "DinoRAG",
  "domain_name": "Dinosaurs",
  "catalog": "main",
  "schema": "default",
  "volume": "dino_pdfs",
  "volume_uri": "/Volumes/main/default/dino_pdfs",
  "project_uri": "/Volumes/main/default/dino_pdfs/project",
  "artifacts_uri": "/Volumes/main/default/dino_pdfs/project/artifacts",
  "index_uri": "/Volumes/main/default/dino_pdfs/project/faiss_index",
  "config_uri": "/Volumes/main/default/dino_pdfs/project/config",
  "mlflow_experiment": "/Shared/Dinosaur_RAG_Experiments",
  "chunk_size": 800,
  "chunk_overlap": 150,
  "top_k": 4,
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "chat_endpoint": "databricks-meta-llama-3-1-8b-instruct",
  "delta_table": "main.default.dinosaur_chunks"
}


In [0]:
class SBERTEmbeddings(Embeddings):
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.model=SentenceTransformer(model_name)
    def embed_documents(self,texts):
        return self.model.encode(texts).tolist()
    def embed_query(self,text):
        return self.model.encode([text])[0].tolist()
embeddings=SBERTEmbeddings(EMBEDDING_MODEL_NAME)
print("Embedding model loaded:", EMBEDDING_MODEL_NAME)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [0]:
LOCAL_INDEX_DIR="/Volumes/main/default/dino_pdfs/project/temp_faiss_runtime"
os.makedirs(LOCAL_INDEX_DIR, exist_ok=True)
for f in dbutils.fs.ls(INDEX_URI):
    if not f.isDir():
        dbutils.fs.cp(f.path, f"{LOCAL_INDEX_DIR}/{f.name}", True)
print(os.listdir(LOCAL_INDEX_DIR))
vectorstore=FAISS.load_local(LOCAL_INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
print("FAISS loaded")

['documents.pkl', 'index.faiss', 'index.pkl']
FAISS loaded


In [0]:
llm=ChatDatabricks(endpoint=CHAT_ENDPOINT, temperature=0.1, max_tokens=500)
print("Chat endpoint ready:", CHAT_ENDPOINT)


Chat endpoint ready: databricks-meta-llama-3-1-8b-instruct


In [0]:
PDF_SYSTEM_PROMPT=("You are a dinosaur domain assistant. Answer only from the provided PDF context. "
                   "Do not use outside knowledge. If the answer is not supported by the context, say: "
                   "'I could not find enough information in the PDF knowledge base.' Be concise and factual.")
EXTERNAL_SYSTEM_PROMPT=("You are a helpful assistant with general dinosaur knowledge. "
                        "The internal dinosaur PDF knowledge base did not provide enough information. "
                        "Answer the question using general knowledge only. Do not claim that the answer came from the PDFs.")
chat_history=[]
def format_history(history,max_turns=3):
    recent=history[-max_turns:]
    if not recent: return "No previous conversation."
    return "\n".join([f"User: {x['user']}\nAssistant: {x['assistant']}" for x in recent])
def retrieve_with_scores(question,k=4):
    results=vectorstore.similarity_search_with_score(question,k=k)
    docs=[d for d,s in results]
    scores=[float(s) for d,s in results]
    context="\n\n".join([d.page_content for d in docs])
    return docs,scores,context
def has_strong_pdf_context(docs,context,scores,min_docs=2,min_total_chars=300,max_avg_distance=1.6):
    if not docs or len(docs)<min_docs: return False
    if len(context.strip())<min_total_chars: return False
    if not scores: return False
    return (sum(scores)/len(scores)) <= max_avg_distance
def answer_from_pdf(question,docs,history_text):
    context="\n\n".join([d.page_content for d in docs])
    prompt=f"{PDF_SYSTEM_PROMPT}\n\nConversation history:\n{history_text}\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:\n"
    answer=llm.invoke(prompt).content
    sources=[{"source_file":d.metadata.get("source_file"),"page":d.metadata.get("page"),"chunk_id":d.metadata.get("chunk_id")} for d in docs]
    return {"mode":"pdf","label":"Answer source: PDF Knowledge Base","answer":answer,"sources":sources}
def answer_from_external(question,history_text):
    prompt=f"{EXTERNAL_SYSTEM_PROMPT}\n\nConversation history:\n{history_text}\n\nQuestion:\n{question}\n\nAnswer:\n"
    answer=llm.invoke(prompt).content
    return {"mode":"external","label":"Answer source: External Knowledge","answer":answer,"sources":[]}
def ask_dino_bot(question,k=None,save_memory=True):
    if k is None: k=DEFAULT_TOP_K
    docs,scores,context=retrieve_with_scores(question,k=k)
    history_text=format_history(chat_history)
    result=answer_from_pdf(question,docs,history_text) if has_strong_pdf_context(docs,context,scores) else answer_from_external(question,history_text)
    result["retrieval_scores"]=scores
    if save_memory: chat_history.append({"user":question,"assistant":result["answer"]})
    return result


In [0]:
def format_sources(sources):
    if not sources: return "No PDF sources used."
    return "\n".join([f"- {s['source_file']} | page {s['page']} | chunk {s['chunk_id']}" for s in sources])
def respond(message, history):
    if history is None: history=[]
    try:
        result=ask_dino_bot(message, save_memory=True)
        final_text=f"{result['label']}\n\n{result['answer']}\n\nSources:\n{format_sources(result['sources'])}"
    except Exception as e:
        final_text="Application error:\n"+str(e)+"\n\n"+traceback.format_exc()
    history.append({"role":"user","content":message})
    history.append({"role":"assistant","content":final_text})
    return history, ""
def clear_chat():
    global chat_history
    chat_history=[]
    return []


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# Dinosaur RAG Chatbot")
    gr.Markdown("This chatbot first tries the PDF knowledge base and falls back to external knowledge when PDF retrieval is weak.")
    chatbot=gr.Chatbot(height=500)
    msg=gr.Textbox(label="Ask a dinosaur question")
    clear=gr.Button("Clear Chat")
    msg.submit(respond,[msg,chatbot],[chatbot,msg])
    clear.click(clear_chat,None,chatbot)
demo.launch(share=True, debug=True)